# Reproducing Okuno et al. (2025) from dandiset 000727

Paper. Okuno T, Woodward A, Okano H, Hata J (2025). Functional connectivity, structural
connectivity, and inter-individual variability in *Drosophila melanogaster*. eLife reviewed
preprint 107990 version 1, [doi:10.7554/elife.107990.1](https://doi.org/10.7554/elife.107990.1).
The claims file was written from the publisher's HTML of version 1 (main text, Methods, figure
legends and the legends of the Extended Data figures); no PDF was available.

Dandiset. [DANDI:000727](https://dandiarchive.org/dandiset/000727/0.240106.0043), version
0.240106.0043, the version the paper's Methods and Data availability cite. It is also the only
published version; the draft lists the same 18 assets with the same total size.

What the dandiset holds. The whole-brain two-photon imaging of Brezovec et al. that the paper
reuses: 9 flies, one NWB file each, with the raw GCaMP6f and tdTomato volumes (256 x 128 x 49 voxels,
3384 volumes at 1.88 Hz), anatomical stacks, a green series already registered to the FDA space by
Brezovec et al. (high-pass filtered and z-scored), and FicTrac walking data at 50 Hz.

Outside inputs (the claims file says the paper relies on them; both are pinned):
- the paper's code, github.com/takuto-okuno-riken/flywalk at commit 4b92add: the resized FDA
  template, brain mask, the hemibrain neuropil atlas (114 ROIs), the JFRC2010 atlas, the FlyEM
  cf0.8 SC of the 114 ROIs and the walking regressors it computed (data/behavior.mat);
- the paper's Zenodo record [10.5281/zenodo.15532272](https://doi.org/10.5281/zenodo.15532272)
  (version of 2025-05-28, the one the code's README cites; the later version 18151010 postdates
  the preprint): FlyWire sc140 SC of the hemibrain ROIs, the selected-synapse SC matrices, the 99
  null SC matrices of each permutation test, per-ROI transmitter and post-synapse counts.

The SCKm and DistKm ROI atlases (k-means parcellations) are in neither, so every claim on them is
blocked. The analysis streams each fly's raw green series once for FC and once for the GLM.

| Claim | Data | Status |
|---|---|---|
| fig1-poltcomp-optimal | with inputs | blocked: SCKm and DistKm atlases not published |
| fig2b-distkm-smoothing-optimum | with inputs | blocked: DistKm atlases not published |
| figs2-1b-sckm-smoothing | with inputs | blocked: SCKm atlases not published |
| fig2c-smoothing-plus-poltcomp | with inputs | blocked: atlases not published |
| figs2-1d-highpass-degrades | with inputs | blocked: DistKm atlases not published |
| fig2d-log-decline, fig2d-fly-values, fig2d-ten-thousand-rois | with inputs | blocked: atlases not published |
| discussion-twenty-rois | with inputs | blocked: atlases not published |
| fig3d-confidence-threshold | with inputs | blocked: FlyEM SC at 0.5, 0.6, 0.7, 0.9 not published |
| fig3d-flyem-vs-flywire | with inputs | tested |
| fig3e-flyem-higher-most-regions | with inputs | tested |
| fig3g-transmitter-correlation | with inputs | tested |
| fig4e-unsegregated-higher | with inputs | tested |
| fig4f-segregated-lower | with inputs | tested (reading a) |
| fig4i-reciprocal-higher | with inputs | tested |
| figs4-1-flyem-permutation | with inputs | blocked: FlyEM selected and null SC not published |
| fig5b-walking-activation | with inputs | tested |
| fig5b-walking-suppression | with inputs | tested |
| fig3a-postsynapse-counts, fig3c-regional-detection, figs3-1a-regional-count-similarity, fig3f-output-transmitters | elsewhere | tested on the paper's per-ROI tables |
| fig4b-ppssi-distribution | elsewhere | tested in `connectome_okuno_2025.ipynb` |
| figs3-2cd-apl-similarity, fig4a, fig4c, figs4-1ij, fig4h, fig5-wagn | elsewhere | blocked |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time, resource, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ants
import flyfc

warnings.filterwarnings("ignore", category=RuntimeWarning)
T_START = time.time()
print("dandiset", flyfc.DANDISET, "version", flyfc.VERSION)
print("code commit", flyfc.CODE_COMMIT, "zenodo record", flyfc.ZENODO_RECORD)

## Data check

The files, their metadata, and what the analysis reads from them. ANTs registration is given a
seed (`random_seed=1`), but it is not exactly repeatable between runs (not run here: repeated
registrations of one mean image, made while preparing, differed slightly), so numbers that rest
on it can move in the second or third decimal between runs; the conclusions quote this run's
output. No other
random numbers are drawn.

In [ ]:
assets = flyfc.list_assets()
print(len(assets), "NWB files")
meta = []
for a in assets:
    h5 = flyfc.open_nwb(a["s3"])
    g = h5[flyfc.GREEN]
    st = h5["acquisition/TwoPhotonSeriesFunctionalGreen/starting_time"]
    gs = h5["general/optophysiology/ImagingPlaneGCaMP6fFunctional/grid_spacing"][()]
    spd = h5["processing/behavior/FicTrac/SpatialSeriesMovementSpeed"]
    proc = h5["processing/ophys/TwoPhotonSeriesFunctionalGreenProcessed/data"]
    row = dict(subject=a["subject"], size_gb=round(a["size"] / 1e9, 1), green=g.shape, dtype=str(g.dtype),
               rate=round(float(st.attrs["rate"]), 6), spacing_um=tuple(np.round(gs * 1e6, 3)),
               speed_n=spd["data"].shape[0], speed_rate=float(spd["starting_time"].attrs["rate"]),
               speed_desc=spd.attrs["description"], processed=proc.shape,
               sex=h5["general/subject/sex"][()].decode(),
               genotype=h5["general/subject/genotype"][()].decode())
    meta.append(row)
    h5.close()
for r in meta:
    print(r["subject"], r["size_gb"], "GB green", r["green"], r["dtype"], "rate", r["rate"], "Hz spacing", r["spacing_um"],
          "um | speed", r["speed_n"], "samples at", r["speed_rate"], "Hz | processed", r["processed"], "|", r["sex"])
print("genotypes:", sorted({r["genotype"] for r in meta}))
print("speed description:", meta[0]["speed_desc"])
print("imaging duration (s):", sorted({round(r["green"][0] / r["rate"], 1) for r in meta}),
      "behaviour duration (s):", sorted({r["speed_n"] / r["speed_rate"] for r in meta}))

Orientation and registration. The template and the masks come from the paper's code. The
native mean image of the first fly is shown beside the template, and the correlation with the
template after an affine registration is printed for each of the eight axis flips.

In [ ]:
tm, tz = flyfc.code_nifti("template/thresholded_FDACal.nii.gz")
tmpl = ants.from_numpy(tm, spacing=tuple(float(v) / 1000 for v in tz))
brain, _ = flyfc.code_nifti("template/thresholded_FDACal_mask.nii.gz")
brain = brain > 0
masks = flyfc.hemiroi_masks()
names = flyfc.hemiroi_names()
print("template", tm.shape, "voxel (um)", tuple(round(float(v), 3) for v in tz), "brain-mask voxels", int(brain.sum()))
print("hemibrain ROIs", len(masks), "voxels per ROI", min(int(m.sum()) for m in masks), "to", max(int(m.sum()) for m in masks))

h5 = flyfc.open_nwb(assets[0]["s3"])
rd0 = flyfc.ChunkReader(assets[0]["s3"], h5)
T0 = rd0.shape[0]; ct = rd0.chunks[0]
mean0 = rd0.mean_image({0, ct * ((T0 // ct) // 2), ct * ((T0 - 1) // ct)})
h5.close()
flips = flyfc.choose_flip(mean0, tmpl)
for k, v in flips.items():
    print("flip", k, "affine correlation with template", round(v, 3))
fig, ax = plt.subplots(2, 4, figsize=(15, 5))
for i, z in enumerate([6, 18, 30, 42]):
    ax[0, i].imshow(mean0[:, :, z].T, cmap="gray"); ax[0, i].set_title(f"{assets[0]['subject']} native z={z}")
    ax[1, i].imshow(tm[:, :, z].T, cmap="gray"); ax[1, i].set_title(f"template z={z}")
plt.tight_layout(); plt.show()
print("native mean image range", mean0.min().round(1), mean0.max().round(1), "| template range", tm.min(), tm.max().round(3))

In both images the slices near z = 6 hold two small dorsal-anterior structures near the midline and
the slices near z = 42 hold the optic lobes, and the dorsoventral axis runs the same way, so no
flip is applied along y or z. Left and right cannot be told apart from a symmetric image (the
x-flipped and unflipped correlations are nearly equal); no flip is assumed, and the main FC-SC
results are shown with left and right ROIs swapped.

Reduce each fly. Each fly's raw green series is streamed once (about 7 GB per fly) and reduced
to the 114 ROI mean series and the 1000 highest-SD voxels (`flyfc.process_fly`).

In [ ]:
flies = []
for a in assets:
    f = flyfc.process_fly(a, tmpl, masks)
    f.pop("warped"); f.pop("mean")
    flies.append(f)
    print(f["subject"], "read", round(f["mb_read"]), "MB in", round(f["t_total"]), "s | correlation with template",
          {k: round(v, 3) for k, v in f["cc"].items()}, "| ROIs not fully in the field of view",
          int((f["cover"] < 0.999).sum()), "min coverage", round(float(np.nanmin(f["cover"])), 3), flush=True)
print("peak memory (GB)", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "| elapsed (min)", round((time.time() - T_START) / 60, 1))

In [ ]:
# Which ROIs lose coverage, and in which fly
for f in flies:
    low = np.nonzero(f["cover"] < 0.999)[0]
    if low.size:
        print(f["subject"], ", ".join(f"{names[i]} {f['cover'][i]:.2f}" for i in low))
sub_order = [f["subject"] for f in flies]
print("flies in name order:", sub_order)
print("code's subject list [1 2 3 4 5 6 8 9] leaves out the 7th:", sub_order[6])
sel8 = [i - 1 for i in flyfc.SUBJECT_IDS]

Subject order and the behaviour clock. The paper's code holds the per-volume speed and the
walking regressor it computed for its nine flies (`data/behavior.mat`). Computing them from the
dandiset's FicTrac speed, with the code's offset of 104.654892 ms, and comparing fly by fly
checks both the subject order and the alignment of the behaviour to the volumes.

In [ ]:
reg, mscal, th_speed, b_speed = flyfc.walking_regressors([f["speed"] for f in flies])
reg_bin, _, _, _ = flyfc.walking_regressors([f["speed"] for f in flies], binary=True)
beh = flyfc.load_mat(flyfc.code_file("data/behavior.mat"))
M = np.asarray(beh["MScal"], float); M2 = np.asarray(beh["MBcal2"], float)
print("code's MScal", M.shape, "| th (mean SD of speed)", round(th_speed, 5), "b (mean 10th percentile)", round(b_speed, 5))
C = np.array([[np.corrcoef(mscal[i], M[j])[0, 1] for j in range(9)] for i in range(9)])
for i, f in enumerate(flies):
    print(f["subject"], "best-matching code fly", int(np.argmax(C[i])) + 1, "r", round(C[i].max(), 6),
          "| max |speed diff|", f"{np.abs(mscal[i] - M[i]).max():.2e}", "| regressor max |diff|", f"{np.abs(reg[i] - M2[i]).max():.2e}")
print("speed range", round(min(f["speed"].min() for f in flies), 4), "to", round(max(f["speed"].max() for f in flies), 4), "(radians per frame, as the file states)")
print("fraction of volumes at or above b + th (binary walking):", np.round((mscal >= b_speed + th_speed).mean(1), 3))

The behaviour clock is checked against the imaging with a contrast: the brain-wide mean of the ROI
series, after removing the 7 polynomial terms, against the walking regressor, at shifts of the regressor by -60 to 60 volumes. If the
two streams share the clock, the correlation peaks near shift 0 (the regressor is already
convolved with the calcium response).

In [ ]:
shifts = np.arange(-60, 61, 2)
ids_all = np.arange(114)
for i, f in enumerate(flies):
    g = flyfc.residualize(f["R"][ids_all].mean(0)[:, None], flyfc.poly_regressors(f["T"]))[:, 0]
    cc = [np.corrcoef(np.roll(reg[i], s)[70:-70], g[70:-70])[0, 1] for s in shifts]
    k = int(np.argmax(cc))
    print(f["subject"], "peak r", round(cc[k], 3), "at shift", int(shifts[k]), "volumes | r at 0:", round(cc[30], 3),
          "| median |r| at shifts beyond 30:", round(float(np.median(np.abs(np.array(cc)[np.abs(shifts) > 30]))), 3))

Duplicates. Recordings deposited twice would give identical speed traces or mean images.

In [ ]:
sp = np.array([f["speed"] for f in flies])
csp = np.corrcoef(sp)
np.fill_diagonal(csp, np.nan)
print("largest speed-trace correlation between two flies:", round(np.nanmax(csp), 3))
roi = np.array([f["R"].mean(1) for f in flies])
cr = np.corrcoef(roi); np.fill_diagonal(cr, np.nan)
print("largest correlation of mean ROI intensity profiles between two flies:", round(np.nanmax(cr), 3))
print("identical speed traces:", [(i, j) for i in range(9) for j in range(i + 1, 9) if np.array_equal(sp[i], sp[j])])

## FC and the paper's SC

FC per fly is the Pearson correlation of the 114 ROI series, Fisher-z transformed and averaged
over the code's eight flies (M3). Poly-tCompCor is the 7 polynomial terms and 6 tCompCor
components; tCompCor is taken from the 1000 native voxels with the highest temporal SD (an
assumption: the code takes them among template voxels, after warping). SC is the second
("Traced") layer of each connect list (M4).

In [ ]:
em_nc, em_sc, em_ids52 = flyfc.load_sc(flyfc.code_file("results/sc/hemiroi_connectlist.mat"))
fw_nc, fw_sc, ids = flyfc.load_sc(flyfc.zenodo_member("results/sc/hemiroi_fw0sr140_connectlist.mat"))
print("FlyEM SC", em_nc.shape, "| FlyWire SC", fw_nc.shape, "| primary ROIs", len(ids))
print("primary ROIs:", ", ".join(names[i] for i in ids))
print("FlyEM neuron SC over primary ROIs: total", int(em_nc[np.ix_(ids, ids)].sum()), "zeros", int((em_nc[np.ix_(ids, ids)] == 0).sum()))
print("FlyWire neuron SC over primary ROIs: total", int(fw_nc[np.ix_(ids, ids)].sum()), "zeros", int((fw_nc[np.ix_(ids, ids)] == 0).sum()))
series = {n: [flyfc.roi_series(f, n) for f in flies] for n in ["", "poltcomp"]}
mFz = {(n, s): flyfc.mean_zfc([series[n][i] for i in S]) for n in series for s, S in [("8", sel8), ("9", range(9))]}
F = mFz[("poltcomp", "8")]
off = ~np.eye(len(ids), dtype=bool)
print("mean z-FC (Poly-tCompCor, 8 flies) over primary ROI pairs: range", round(F[np.ix_(ids, ids)][off].min(), 3), "to", round(F[np.ix_(ids, ids)][off].max(), 3),
      "| diagonal set to", round(F[ids[0], ids[0]], 3))
for f, s in zip(flies, series["poltcomp"]):
    print(f["subject"], "ROI series SD range", round(s.std(0)[ids].min(), 2), "to", round(s.std(0)[ids].max(), 2))

### fig3d-flyem-vs-flywire

> Against the same FC, FlyEM gives a higher matrix-level FC-SC correlation than FlyWire (0.73
> against 0.68).

63 hemibrain primary ROIs, no smoothing, Poly-tCompCor, SC (neuron), FlyEM cf0.8 and FlyWire
sc140. Reproduced when both values are within tolerance (0.657 to 0.803; 0.612 to 0.748) and FlyEM
is higher.

In [ ]:
lr = {}
for i, n in enumerate(names):
    if "(L)" in n:
        j = names.index(n.replace("(L)", "(R)")) if n.replace("(L)", "(R)") in names else i
    elif "(R)" in n:
        j = names.index(n.replace("(R)", "(L)")) if n.replace("(R)", "(L)") in names else i
    else:
        j = i
    lr[i] = j
perm = np.array([lr[i] for i in range(114)])
print("ROIs with a left-right partner:", int((perm != np.arange(114)).sum()))
ref = flyfc.load_mat(flyfc.zenodo_member("results/auc/poltcomparhemiroi_fw0sr140-fcauc.mat"))
print("paper's own output for this setting (Zenodo auc file, FlyWire): R(7) =", round(float(np.ravel(ref["R"])[6]), 4), "R(9) =", round(float(np.ravel(ref["R"])[8]), 4))
rows = []
for (n, s), Fm in mFz.items():
    for lab, Fx in [("as registered", Fm), ("left-right swapped", Fm[np.ix_(perm, perm)])]:
        rows.append((n or "none", s, lab, flyfc.fcsc_r(em_nc, Fx, ids), flyfc.fcsc_r(fw_nc, Fx, ids),
                     flyfc.fcsc_r(em_sc, Fx, ids), flyfc.fcsc_r(fw_sc, Fx, ids)))
print(f"{'nuisance':9s} {'flies':5s} {'orientation':20s} {'EM neuron':>9s} {'FW neuron':>9s} {'EM syn':>7s} {'FW syn':>7s}")
for r in rows:
    print(f"{r[0]:9s} {r[1]:5s} {r[2]:20s} {r[3]:9.4f} {r[4]:9.4f} {r[5]:7.4f} {r[6]:7.4f}")
signed = [np.corrcoef(flyfc.log_sc(S[np.ix_(ids, ids)]).ravel(), F[np.ix_(ids, ids)].ravel())[0, 1] for S in (em_nc, fw_nc)]
print("signed FC variant (O3), Poly-tCompCor, 8 flies: FlyEM", round(signed[0], 4), "FlyWire", round(signed[1], 4))
r_em, r_fw = rows[4][3], rows[4][4]
print(f"MAIN: FlyEM r = {r_em:.4f} (tolerance 0.657 to 0.803, within: {0.657 <= r_em <= 0.803}); "
      f"FlyWire r = {r_fw:.4f} (0.612 to 0.748, within: {0.612 <= r_fw <= 0.748}); FlyEM higher: {r_em > r_fw}")
# per fly
for i, f in enumerate(flies):
    Fi = flyfc.mean_zfc([series["poltcomp"][i]])
    print(f["subject"], "single-fly FC: FlyEM", round(flyfc.fcsc_r(em_nc, Fi, ids), 4), "FlyWire", round(flyfc.fcsc_r(fw_nc, Fi, ids), 4))

In [ ]:
a = flyfc.log_sc(em_nc[np.ix_(ids, ids)]).ravel(); b = np.abs(F[np.ix_(ids, ids)]).ravel()
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(a, b, s=3); ax[0].set_xlabel("log10 SC neuron (FlyEM)"); ax[0].set_ylabel("|mean z-FC|")
a2 = flyfc.log_sc(fw_nc[np.ix_(ids, ids)]).ravel()
ax[1].scatter(a2, b, s=3, c="C1"); ax[1].set_xlabel("log10 SC neuron (FlyWire)")
plt.tight_layout(); plt.show()
print("plotted ranges: log10 SC FlyEM", a.min(), "to", round(a.max(), 2), "| FlyWire", a2.min(), "to", round(a2.max(), 2), "| |mean z-FC|", round(b.min(), 3), "to", round(b.max(), 3), "(all points shown)")

**Result: FlyEM gives the higher FC-SC correlation, but both values are below the paper's.**
With the code's eight flies and Poly-tCompCor, r is 0.616 for FlyEM and 0.575 for FlyWire, below
the lower bounds of the tolerances (0.657 and 0.612), so the claim is partially reproduced. FlyEM
is higher in every as-registered variant (no nuisance removal 0.638 against 0.606, nine flies
0.608 against 0.568, signed FC the same as the main values). The paper's own output file for the
FlyWire setting gives 0.684 with the same SC, so the gap is in this attempt's FC, not in the SC.
Swapping left and right lowers both correlations (FlyEM 0.531, FlyWire 0.520), which supports the
orientation assumed.

### fig3e-flyem-higher-most-regions

> Per region, FlyEM gives the higher FC-SC correlation in 58 of the 63 primary neuropils, FlyWire in 5.

Per-region correlation (code's roiR column 7), settings of fig3d. Reproduced when the count is
within 58 +- 5.8.

In [ ]:
counts = {}
for (n, s), Fm in mFz.items():
    for lab, Fx in [("as registered", Fm), ("left-right swapped", Fm[np.ix_(perm, perm)])]:
        re_ = flyfc.fcsc_roi_r(em_nc, Fx, ids); rw_ = flyfc.fcsc_roi_r(fw_nc, Fx, ids)
        counts[(n or "none", s, lab)] = int((re_ > rw_).sum())
for k, v in counts.items():
    print(k, "FlyEM higher in", v, "of", len(ids))
roi_em = flyfc.fcsc_roi_r(em_nc, F, ids); roi_fw = flyfc.fcsc_roi_r(fw_nc, F, ids)
n_em = int((roi_em > roi_fw).sum())
print("MAIN (Poly-tCompCor, 8 flies):", n_em, "of 63; within 52.2 to 63.8:", 52.2 <= n_em <= 63.8)
print("regions where FlyWire is higher:", ", ".join(f"{names[ids[i]]} ({roi_em[i]:.2f} vs {roi_fw[i]:.2f})" for i in np.nonzero(roi_fw >= roi_em)[0]))
print("per-region r range: FlyEM", round(roi_em.min(), 3), "to", round(roi_em.max(), 3), "| FlyWire", round(roi_fw.min(), 3), "to", round(roi_fw.max(), 3))
pref = np.ravel(np.asarray(ref["roiR"])[:, 6]) if np.asarray(ref["roiR"]).shape[0] == 63 else np.ravel(np.asarray(ref["roiR"])[6])
print("paper's own per-region FlyWire values (Zenodo auc file) against this attempt's: r =", round(np.corrcoef(pref, roi_fw)[0, 1], 3))

**Result: reproduced, resting on the left-right orientation.** FlyEM is higher in 55 of 63
regions under the main setting (tolerance 52.2 to 63.8) and in 55 with nine flies. With left and
right swapped the count is 49, outside the tolerance, and without nuisance removal it is 50 (48
with nine flies); the swapped orientation also gives the lower matrix-level FC-SC correlation
(fig3d), so the orientation used is the better supported one. This attempt's per-region FlyWire
values correlate with the paper's own (Zenodo) at r = 0.72.

### fig3g-transmitter-correlation

> Across regions, the FC-SC correlation rises with the share of cholinergic output neurons
> (r = 0.39, p = 0.0013) and falls with the share of GABAergic ones (r = -0.25, p = 0.046).

The per-region FlyWire sc140 FC-SC correlation against each region's share of output neurons of
each transmitter (plotStructConnectivity.m showNeuralTransmitter: counts divided by the ROI's
total over the seven classes, including unknown). Reproduced when both r are within tolerance
(0.351 to 0.429; -0.275 to -0.225) and both p < 0.05.

In [ ]:
tr = flyfc.load_mat(flyfc.zenodo_member("results/sc/hemiroi_fw0sr140_transmitter.mat"))
outN = np.asarray(tr["outNTypes"], float)
print("outNTypes", outN.shape)
share = outN / outN.sum(1, keepdims=True)
labs = ["Unknown", "DA", "SER", "GABA", "GLUT", "ACH", "OCT"]
res = {}
for lab_, Fx in [("main", F), ("no nuisance", mFz[("", "8")]), ("nine flies", mFz[("poltcomp", "9")]), ("left-right swapped", F[np.ix_(perm, perm)])]:
    rr = flyfc.fcsc_roi_r(fw_nc, Fx, ids)
    out = []
    for k in range(1, 7):
        r, p = stats.pearsonr(rr, share[ids, k]); out.append(f"{labs[k]} r={r:.3f} p={p:.3g}")
        res[(lab_, labs[k])] = (r, p)
    print(lab_, "|", "; ".join(out))
if np.asarray(ref["roiR"]).size:
    out = []
    for k in range(1, 7):
        r, p = stats.pearsonr(pref, share[ids, k]); out.append(f"{labs[k]} r={r:.3f} p={p:.3g}")
    print("paper's own per-region values (Zenodo auc file) |", "; ".join(out))
r_a, p_a = res[("main", "ACH")]; r_g, p_g = res[("main", "GABA")]
print(f"MAIN: ACh r = {r_a:.3f} (p = {p_a:.3g}); GABA r = {r_g:.3f} (p = {p_g:.3g})")

**Result: not reproduced.** With this attempt's FC the per-region FC-SC correlation does not rise
with the cholinergic share (r = 0.013, p = 0.92) and its relation to the GABAergic share is not
significant (r = -0.157, p = 0.22); no variant gives a significant ACh or GABA correlation. The
paper's own per-region values from its Zenodo file, run through the same transmitter table, give
the paper's numbers (ACh r = 0.395, p = 0.0014; GABA r = -0.252, p = 0.047), so the difference
comes from the FC computed here, not from the transmitter table.

### Permutation tests: fig4e-unsegregated-higher, fig4f-segregated-lower, fig4i-reciprocal-higher

> The SC matrix of unsegregated synapses (cPPSSI 0 to 0.1) has a significantly higher FC-SC
> correlation than 99 random synapse samples of matched size, for both neuron and synapse SC.

> The SC matrix of highly segregated synapses (cPPSSI 0.9 to 1) has a significantly lower FC-SC
> correlation than random synapse samples.

> The SC matrix of reciprocal synapses within 2 um has a significantly higher FC-SC correlation
> than random synapse samples.

M6: the selected SC against the 99 null SC matrices the code pairs with it (rd185-20, rd1560-80,
rd140-15), a normal fitted to the null values, two-tailed p, threshold 0.05/6. The null totals are
checked against the selected total in the same way.

In [ ]:
perm_res = {}
for claim, tgt, null in [("fig4e-unsegregated-higher", "sp10db3000mi1_only1", "rd185-20"),
                         ("fig4f-segregated-lower", "sp90db3000mi1", "rd1560-80"),
                         ("fig4i-reciprocal-higher", "rc20_only1", "rd140-15")]:
    nc, sc, _ = flyfc.load_sc(flyfc.zenodo_member(f"results/sc/hemiroi_fw0sr140_{tgt}_connectlist.mat"))
    nulls = [flyfc.load_sc(flyfc.zenodo_member(f"results/sc/hemiroi_fw0sr140_{null}-{k}_connectlist.mat")) for k in range(1, 100)]
    d = dict(tot=sc[np.ix_(ids, ids)].sum(), ntot=nc[np.ix_(ids, ids)].sum(),
             null_tot=np.array([b[np.ix_(ids, ids)].sum() for a, b, _ in nulls]))
    for lab_, Fx in [("main", F), ("no nuisance", mFz[("", "8")]), ("nine flies", mFz[("poltcomp", "9")]), ("left-right swapped", F[np.ix_(perm, perm)])]:
        r7 = flyfc.fcsc_r(nc, Fx, ids); r9 = flyfc.fcsc_r(sc, Fx, ids)
        n7 = np.array([flyfc.fcsc_r(a, Fx, ids) for a, b, _ in nulls]); n9 = np.array([flyfc.fcsc_r(b, Fx, ids) for a, b, _ in nulls])
        d[lab_] = (r7, n7, r9, n9)
    perm_res[claim] = d
    print(claim, "| target", tgt, "synapse total", int(d["tot"]), "neuron total", int(d["ntot"]), "| null", null, "n =", len(nulls),
          "synapse totals mean", int(d["null_tot"].mean()), "SD", int(d["null_tot"].std(ddof=1)),
          "| totals test p =", round(flyfc.normal_p(d["tot"], d["null_tot"])[0], 3), flush=True)

def show_perm(claim):
    d = perm_res[claim]
    for lab_ in ["main", "no nuisance", "nine flies", "left-right swapped"]:
        r7, n7, r9, n9 = d[lab_]
        p7, m7, s7 = flyfc.normal_p(r7, n7); p9, m9, s9 = flyfc.normal_p(r9, n9)
        print(f"{lab_:19s} neuron r={r7:.4f} null {m7:.4f}+-{s7:.4f} z={(r7 - m7) / s7:+.1f} p={p7:.3g} "
              f"(null range {n7.min():.4f} to {n7.max():.4f}) | synapse r={r9:.4f} null {m9:.4f}+-{s9:.4f} z={(r9 - m9) / s9:+.1f} p={p9:.3g}")

In [ ]:
show_perm("fig4e-unsegregated-higher")
d = perm_res["fig4e-unsegregated-higher"]
print("expected total 2,373,100 +- 237,310; measured", int(d["tot"]))
r7, n7, r9, n9 = d["main"]
print("MAIN: neuron above null with p < 0.05/6:", r7 > n7.mean() and flyfc.normal_p(r7, n7)[0] < 0.05 / 6,
      "| synapse:", r9 > n9.mean() and flyfc.normal_p(r9, n9)[0] < 0.05 / 6)

**Result: reproduced.** Under the main setting the unsegregated SC has a higher FC-SC correlation
than the null for neuron SC (0.634 against 0.593 +- 0.002, p = 1e-153) and synapse SC (0.583
against 0.564 +- 0.002, p = 2e-17). The selected total is 2,373,104 (expected 2,373,100) and the
null totals do not differ from it (p = 0.832, as in the paper). Without nuisance removal the
direction reverses (both below the null, p = 3e-6 and 6e-15), so the result depends on
Poly-tCompCor; with nine flies and with left and right swapped both are still above the null
(swapped synapse SC p = 0.0021, below 0.05/6).

In [ ]:
show_perm("fig4f-segregated-lower")
r7, n7, r9, n9 = perm_res["fig4f-segregated-lower"]["main"]
print("MAIN (reading a, the code's target): neuron below null with p < 0.05/6:", r7 < n7.mean() and flyfc.normal_p(r7, n7)[0] < 0.05 / 6,
      "| synapse:", r9 < n9.mean() and flyfc.normal_p(r9, n9)[0] < 0.05 / 6)
print("reading b (both sides at 0.9 or more) needs an SC rebuilt from the synapse table; not run here")

**Result: reproduced under reading (a).** The segregated SC (the code's target) has a lower FC-SC
correlation than the null for neuron SC (0.574 against 0.578 +- 0.001, p = 4e-8) and synapse SC
(0.524 against 0.540 +- 0.001, p = 6e-35). The same holds with nine flies and with left and right
swapped; without nuisance removal the neuron SC is not different from the null (p = 0.48).
Reading (b) was not run.

In [ ]:
show_perm("fig4i-reciprocal-higher")
r7, n7, r9, n9 = perm_res["fig4i-reciprocal-higher"]["main"]
print("MAIN: neuron above null with p < 0.05/6:", r7 > n7.mean() and flyfc.normal_p(r7, n7)[0] < 0.05 / 6,
      "| synapse:", r9 > n9.mean() and flyfc.normal_p(r9, n9)[0] < 0.05 / 6)

**Result: reproduced.** The reciprocal-synapse SC has a higher FC-SC correlation than the null for
neuron SC (0.636 against 0.596 +- 0.002, p = 2e-143) and synapse SC (0.577 against 0.568 +- 0.002,
p = 3e-6, below 0.05/6 = 0.0083). With nine flies both still hold; without nuisance removal the
synapse SC is below the null (p = 0.0034), and with left and right swapped it is below the null
without reaching 0.05/6 (p = 0.0195).

## GLM of walking: fig5b-walking-activation and fig5b-walking-suppression

> In the group GLM of walking, the LAL, FB, IPS and GNG neuropils show strong activity.

> The Wedge and AVLP neuropils are inactive or strongly suppressed during walking.

M8: the code's graded walking regressor (reproduced exactly above), 7 polynomial terms and 6
tCompCor components; smoothing FWHM 8 voxels; second level on the walking and tCompCor betas of
the nine flies (glmFlyMB2nd.m), height threshold p < 0.001. Assumptions: OLS without the Tukey-taper
pre-whitening, betas mapped to the template and smoothed there (equal to fitting the warped,
smoothed data under OLS), tCompCor from native voxels. The binary regressor (O1) is a variant.
Cluster extent: the code's README prints k = 167.9 voxels as its extent threshold, which is used
here (18-connectivity, SPM's default).

In [ ]:
t0 = time.time()
betas = []
for a, f, r1, r2 in zip(assets, flies, reg, reg_bin):
    betas.append(flyfc.glm_betas(a, f, [r1, r2]))
    print(f["subject"], "betas done", round(time.time() - t0), "s", flush=True)
shp = brain.shape
glm = {}
for vi, lab_ in enumerate(["graded", "binary"]):
    bs = []
    for b in betas:
        bb = b[vi].reshape(7, *shp)
        bs.append(np.stack([flyfc.glm_smooth(bb[k]) for k in range(7)]).reshape(7, -1)[:, brain.ravel()])
    t, df = flyfc.second_level(bs)
    T = np.zeros(shp, np.float32); T[brain] = t
    glm[lab_] = (T, df)
    th = stats.t.isf(0.001, df)
    lp, sp_ = flyfc.clusters(T > th); ln, sn = flyfc.clusters(T < -th)
    print(lab_, "df", df, "height threshold t >", round(th, 4), "| t range", round(float(t.min()), 2), "to", round(float(t.max()), 2),
          "| positive clusters >= 168 voxels:", sorted(int(s) for s in sp_ if s >= 168), "| negative clusters (any size):", sorted(int(s) for s in sn)[-5:])
del betas
jf, _ = flyfc.code_nifti("atlas/jrc2018f_2010roiCal_invFDACal.nii.gz")
jlab = {}
for line in flyfc.code_file("atlas/jfrc2010-roi-itksnap.txt").decode().splitlines():
    if line.startswith("#") or not line.strip():
        continue
    p = line.split("\t")
    jlab[int(p[0])] = p[-1].strip().strip('"')
print("JFRC2010 labels in the atlas:", len(np.unique(jf[jf > 0])), "| peak memory (GB)", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2))

In [ ]:
def neuropil_table(T, th, atlas):
    rows = []
    if atlas == "hemibrain":
        for i in ids:
            m = masks[i] & brain
            if m.sum():
                rows.append((names[i], m))
    else:
        for k in np.unique(jf[jf > 0]).astype(int):
            m = (jf == k) & brain
            if m.sum():
                rows.append((f"{jlab.get(k, k)} [{k}]", m))
    out = []
    for nm, m in rows:
        out.append((nm, float(T[m].mean()), float(T[m].max()), float(T[m].min()), float((T[m] > th).mean()), float((T[m] < -th).mean()), int(m.sum())))
    return out

targets_hb = {"LAL": ["LAL(R)", "LAL(L)"], "FB": ["FB"], "IPS": ["IPS(R)"], "GNG": ["GNG"]}
targets_jf = {"LAL": ["lateral accessory lobe"], "FB": ["fan-shaped body"], "IPS": ["inferior posterior slope"], "GNG": ["adult gnathal ganglion"]}
act = {}
for lab_ in ["graded", "binary"]:
    T, df = glm[lab_]; th = stats.t.isf(0.001, df); tmax = float(T.max())
    lp, sp_ = flyfc.clusters(T > th)
    big = np.isin(lp, 1 + np.nonzero(sp_ >= 168)[0])
    for atlas, tg in [("hemibrain", targets_hb), ("JFRC2010", targets_jf)]:
        tab = neuropil_table(T, th, atlas)
        order = sorted(tab, key=lambda r: -r[1])
        top = {r[0] for r in order[: int(np.ceil(len(order) / 3))]}
        for key, lst in tg.items():
            hits = [r for r in tab if any(r[0].startswith(x) for x in lst)]
            ra = any((big & masks[names.index(r[0])]).any() and r[2] >= tmax / 2 for r in hits) if atlas == "hemibrain" else \
                 any(r[2] >= tmax / 2 for r in hits)
            rb = any(r[0] in top for r in hits)
            act[(lab_, atlas, key)] = (ra, rb)
            print(lab_, atlas, key, "|", "; ".join(f"{r[0]} mean t {r[1]:.2f} max {r[2]:.2f} rank {1 + [o[0] for o in order].index(r[0])}/{len(order)}" for r in hits),
                  "| reading a:", ra, "reading b:", rb)
    print(lab_, "brain-wide max t", round(tmax, 2))

In [ ]:
T, df = glm["graded"]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for i, z in enumerate([17, 21]):
    im = ax[i].imshow(T[:, :, z].T, cmap="RdBu_r", vmin=-15, vmax=15); ax[i].set_title(f"second-level t, z={z}")
plt.colorbar(im, ax=ax); plt.show()
print("t range in the shown slices", round(float(T[:, :, [17, 21]].min()), 2), "to", round(float(T[:, :, [17, 21]].max()), 2), "; colour scale -15 to 15,",
      int((np.abs(T[:, :, [17, 21]]) > 15).sum()), "voxels beyond it")

**Result: reproduced.** The second level gives one large positive cluster (399,973 voxels; the
code's README prints 416,801) with maximum t = 16.1 (README 14.7). LAL, FB, IPS and GNG all lie
in it with peak t above half the maximum (reading a) under both atlases and both regressors.
Reading (b) fails for FB (rank 26 of 63 by mean t with the hemibrain atlas, 31 of 75 with JFRC2010)
and, with JFRC2010, for LAL; the claim holds under reading (a), as the criterion allows.

In [ ]:
for lab_ in ["graded", "binary"]:
    T, df = glm[lab_]; th = stats.t.isf(0.001, df)
    ln, sn = flyfc.clusters(T < -th)
    print(lab_, "voxels below -", round(th, 3), ":", int((T < -th).sum()), "| negative clusters:", len(sn), "| minimum t", round(float(T.min()), 2))
    for atlas, lst in [("hemibrain", ["WED", "AVLP"]), ("JFRC2010", ["wedge", "anterior ventrolateral protocerebrum"])]:
        tab = neuropil_table(T, th, atlas)
        for r in tab:
            if any(r[0].startswith(x) for x in lst):
                print("  ", lab_, atlas, r[0], f"mean t {r[1]:.2f}, min {r[3]:.2f}, max {r[2]:.2f}, positive-significant fraction {r[4]:.2f}, negative-significant fraction {r[5]:.2f}, voxels {r[6]}")
# Variant: the same with the code's subject list (8 flies), and the lowest t in the brain
T, df = glm["graded"]
lo = np.unravel_index(np.argmin(np.where(brain, T, np.inf)), T.shape)
print("lowest t in the brain mask:", round(float(T[lo]), 2), "at voxel", tuple(int(v) for v in lo), "(the README's negative peaks are at (82,78,12) and (174,80,15), 1-based)")
for v in [(81, 77, 11), (173, 79, 14)]:
    print("t at README negative peak", tuple(x + 1 for x in v), "=", round(float(T[v]), 2))

**Result: not reproduced.** No voxel in the brain mask falls below the negative height threshold
(the lowest t is -0.82), so there is no negative cluster. WED and AVLP are positive: 85 to 100% of
their voxels are above the positive threshold under both atlases and both regressors, and the
README's two negative peaks have t = 3.28 and 4.09 here.

## Connectome-only claims (data elsewhere)

These are tested on the paper's per-ROI tables in its Zenodo record (`hemiroi_postsyncount.mat`,
`hemiroi_fw0sr140_transmitter.mat`), which the paper computed from the FlyEM and FlyWire synapses
in the template; recounting from the raw connectomes was not done (a deviation).

In [ ]:
ps = flyfc.load_mat(flyfc.zenodo_member("results/sc/hemiroi_postsyncount.mat"))
for k, v in ps.items():
    print(k, np.asarray(v).shape, np.ravel(v)[:8] if np.asarray(v).size < 10 else "")
hbS = np.asarray(ps["hbS"], float); fwS = np.asarray(ps["fwS"], float)
confThs = np.ravel(ps["confThs"]); scoreThs = np.ravel(ps["scoreThs"])
pid = np.asarray(ps["primaryIds"]).ravel().astype(int) - 1
if hbS.shape[0] != 114: hbS = hbS.T
if fwS.shape[0] != 114: fwS = fwS.T
for j, c in enumerate(confThs):
    print(f"FlyEM confidence {c}: post-synapses over primary ROIs {hbS[pid, j].sum() / 1e6:.2f} million")
for j, c in enumerate(scoreThs):
    print(f"FlyWire cleft score {c}: post-synapses over primary ROIs {fwS[pid, j].sum() / 1e6:.2f} million")

**Result: partially reproduced.** FlyWire has 18.83 million post-synapses at cleft score 50
(expected 18.8) and 11.24 million at 140 (expected 11.4), close to FlyEM's 11.57 million at
confidence 0.8. The table holds FlyEM only from confidence 0.5 (18.12 million), so the 21.4
million at confidence 0 cannot be checked.

In [ ]:
jem = int(np.argmin(np.abs(confThs - (0.8 if confThs.max() <= 1 else 80))))
jfw = int(np.argmin(np.abs(scoreThs - 140)))
print("FlyEM column", confThs[jem], "FlyWire column", scoreThs[jfw])
nm = [names[i] for i in pid]
def cmp(n):
    i = pid[nm.index(n)]
    return hbS[i, jem], fwS[i, jfw]
mb = [n for n in nm if any(n.startswith(x) for x in ["aL", "a'L", "bL", "b'L", "gL", "CA(", "PED", "MB"])]
print("mushroom body ROIs among the primary ROIs:", mb)
for grp, lst in [("aL, bL", [n for n in nm if n.startswith("aL") or n.startswith("bL")]), ("all MB", mb),
                 ("central complex", [n for n in nm if n.split("(")[0] in ("EB", "FB", "NO", "PB")]),
                 ("optic lobe", [n for n in nm if n.split("(")[0] in ("LO", "ME")]), ("periesophageal", [n for n in nm if n.split("(")[0] in ("CAN", "FLA")])]:
    print(grp, "|", "; ".join(f"{n}: FlyEM {cmp(n)[0]:.0f} FlyWire {cmp(n)[1]:.0f}" for n in lst))

**Result: reproduced.** FlyEM has more post-synapses in aL and bL and in every mushroom body ROI,
and in EB, FB, NO and PB; FlyWire has more in LO and ME and in CAN and FLA.

In [ ]:
x, y = hbS[pid, jem], fwS[pid, jfw]
print("Pearson r", round(stats.pearsonr(x, y)[0], 4), "| Spearman r", round(stats.spearmanr(x, y)[0], 4), "| Pearson r of log10", round(stats.pearsonr(np.log10(x), np.log10(y))[0], 4),
      "| expected 0.778, 0.9, 0.919")

**Result: reproduced.** Pearson r = 0.778, Spearman r = 0.900 and Pearson r on log10 counts =
0.919, as in the paper.

In [ ]:
sh = share[ids] * 100
for k, lab_ in enumerate(labs):
    print(f"{lab_:8s} share of output neurons over the 63 primary ROIs: mean {sh[:, k].mean():.1f}% SD {sh[:, k].std(ddof=1):.1f}% (SD with ddof 0: {sh[:, k].std():.1f}%)")
print("expected ACh 62.8 +- 14.2, GLUT 15.5 +- 9.2, GABA 11.4 +- 8.2")

**Result: reproduced.** Acetylcholine 62.8% +- 14.2%, glutamate 15.5% +- 9.2% and GABA 11.4% +-
8.2%, as in the paper.

## Summary

| Claim | Outcome |
|---|---|
| fig3d-flyem-vs-flywire | partially reproduced: FlyEM higher, both r below tolerance |
| fig3e-flyem-higher-most-regions | reproduced (rests on the left-right orientation) |
| fig3g-transmitter-correlation | not reproduced |
| fig4e-unsegregated-higher | reproduced |
| fig4f-segregated-lower | reproduced (reading a) |
| fig4i-reciprocal-higher | reproduced |
| fig5b-walking-activation | reproduced |
| fig5b-walking-suppression | not reproduced |
| fig3a-postsynapse-counts (elsewhere) | partially reproduced: FlyEM at confidence 0 not in the table |
| fig3c, figs3-1a, fig3f (elsewhere) | reproduced, on the paper's per-ROI tables |
| all SCKm and DistKm claims, fig3d-confidence-threshold, figs4-1-flyem-permutation | blocked: inputs not published |
| fig4b-ppssi-distribution (elsewhere, companion notebook) | partially reproduced: 138,144 neurons have a PPSSI, not 139,255 |
| other connectome-only claims | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. No SPM12 motion or slice-timing correction; the
mean green image is registered to the template with ANTs SyN in Python and the ROI means are
computed through the registration's linear map without warping each volume; tCompCor is taken from
native voxels; the GLM has no Tukey-taper pre-whitening and is smoothed on the beta maps. The FC
these give has a lower FC-SC correlation than the paper's own output files (0.575 against 0.684 for
FlyWire), which bears on the claims that compare FC-SC values across regions. The registration is
not exactly repeatable, so numbers can move slightly between runs.

What was found about the dataset. The files match their description (9 flies, 3384 volumes at
1.8797 Hz, 50 Hz FicTrac, 1800 s), the walking regressor of the paper's code is reproduced to
1e-5 from the files in name order, and the seventh fly in name order (fly-100) has the lowest
coverage of a hemibrain ROI (AL(L) 65% inside the field of view in this run), which fits the
paper's reason for leaving one fly out; fly-105 also has ROIs partly outside. The
`dimension` attribute of the functional series is [128, 256, 49] while the data are stored as
(time, 256, 128, 49). The ROI atlases of the paper's main analyses (SCKm, DistKm) are not
published with the code or on Zenodo.

In [ ]:
print("total runtime (min)", round((time.time() - T_START) / 60, 1), "| peak memory (GB)", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2))